In [1]:
import torch, torchaudio

knn_vc = torch.hub.load('bshall/knn-vc', 'knn_vc', prematched=True, trust_repo=True, pretrained=True)
# Or, if you would like the vocoder trained not using prematched data, set prematched=False.

Using cache found in C:\Users\user/.cache\torch\hub\bshall_knn-vc_master
c:\pyproject1\PO3_objectodyssey\audio_module\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...
[HiFiGAN] Generator loaded with 16,523,393 parameters.
WavLM-Large loaded with 315,453,120 parameters.


In [6]:
import torch
import soundfile as sf

# torchaudio.load를 soundfile로 몽키패치 (knn-vc 로드 전에 실행)
import torchaudio
def sf_load(path, normalize=True, **kwargs):
    wav, sr = sf.read(str(path), dtype="float32", always_2d=True)
    wav = wav.T  # (channels, samples)
    tensor = torch.from_numpy(wav)
    if normalize:
        tensor = tensor / (tensor.abs().max() + 1e-8)
    return tensor, sr

torchaudio.load = sf_load  # 패치 먼저

# 그 다음 knn-vc 로드
matcher = torch.hub.load("bshall/knn-vc", "knn_vc", prematched=True, trust_repo=True)
target_wavs = matcher.get_matching_set(["./audios/default_women.wav"])

Using cache found in C:\Users\user/.cache\torch\hub\bshall_knn-vc_master
c:\pyproject1\PO3_objectodyssey\audio_module\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...
[HiFiGAN] Generator loaded with 16,523,393 parameters.
WavLM-Large loaded with 315,453,120 parameters.
resample 24000 to 16000 in ./audios/default_women.wav


In [9]:
# 감정 음성 특징 추출
query_seq = matcher.get_features("./audios/fairy_tale_combined.wav")

# # 타겟 화자 목소리로 변환
#out_wav = matcher.match(query_seq, target_wavs, topk=4) #노이즈가 많음

out_wav = matcher.match(query_seq, target_wavs, topk=4)  # 음색 약해짐
import noisereduce as nr
import numpy as np

wav_np = out_wav.numpy()

# 노이즈 제거
reduced = nr.reduce_noise(
    y=wav_np,
    sr=matcher.sr,
    stationary=False,   # 비정상 노이즈에 효과적
    prop_decrease=0.8,  # 노이즈 감소 강도 (0~1)
)

sf.write("./audios/fairy_tale_combined_conversioned.wav", reduced, matcher.sr)

# # 저장
# sf.write("./audios/output_final.wav", out_wav.numpy(), matcher.sr)

resample 24000 to 16000 in ./audios/fairy_tale_combined.wav
